# Эксперимент 1 — нужны ли skip-connection'ы на высоких разрешениях

**Кому:** команде. Ноутбук воспроизводим целиком, менять ничего не нужно.
**Время:** ~2.5 часа на одно плечо на RTX 3070. Плечо A, возможно, уже обучено.

---

## Гипотеза

Базовая модель — `smp.Unet` с энкодером `tu-convnext_tiny`. У ConvNeXt стем
режет вход сразу в 4 раза (patchify 4x4), поэтому карты признаков на stride 2
у него **не существует**: пирамида выглядит как `[3, 0, 96, 192, 384, 768]`,
где ноль — заглушка, которую подставляет smp.

Следствие: **два последних блока U-Net-декодера работают вообще без
skip-connection**. Путь от 128x128 до 512x512 модель проходит чистым апсемплом,
не получая от энкодера ни одного признака.

Для нашей задачи это подозрительно. Границы вставленного объекта и следы
блендинга — высокочастотный сигнал, который живёт именно в ранних слоях. Если
восстанавливать его не из чего, страдает `Dice_pos`, а с ним и половина AIC.

У ResNet34 первая свёртка имеет stride 2, и карта на этом разрешении есть:
пирамида `[3, 64, 64, 128, 256, 512]`.

## Что проверяем

| Плечо | Конфигурация | Отличие |
|---|---|---|
| **A** | `unet` + `tu-convnext_tiny` | базовая, два блока без скипов |
| **B** | `unet` + `tu-resnet34` | **единственное изменение** — энкодер |
| **C** (опционально) | `fpn` + `tu-convnext_tiny` | тот же энкодер, архитектура без опоры на ранние скипы |

Всё остальное совпадает: фолд, сид, длина эпохи, число эпох, аугментации,
лосс, расписание LR.

## Честная оговорка про интерпретацию

Плечо B меняет не только наличие скипов — у ResNet34 другой объём параметров и
другое предобучение. Само по себе «B лучше A» ещё не доказывает гипотезу.

Для этого и нужно плечо C: оно оставляет энкодер прежним и убирает зависимость
от ранних скипов через архитектуру. Логика чтения результата:

* B > A **и** C > A — гипотеза про скипы подтверждается;
* B > A, но C ≈ A — дело скорее в самом энкодере, а не в скипах;
* B ≈ A — гипотеза не подтверждается, ищем причину в другом месте.

---
## 1. Настройка окружения

Ячейка находит корень проекта сама, поэтому ноутбук можно запускать откуда угодно.

In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

# ищем корень проекта по наличию cli.py
ROOT = Path.cwd()
while not (ROOT / "cli.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "cli.py").exists(), "не нашёл корень проекта — запусти ноутбук внутри репозитория"
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import experimental_tools_beliy_russak as etbr  # до torch: чинит конфликт OpenMP

import pandas as pd
import torch

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

print("корень проекта:", ROOT)
print("интерпретатор :", sys.executable)
print("torch         :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("gpu           :", props.name, f"{props.total_memory / 1e9:.1f} ГБ")

In [ ]:
def run_cli(args, live=True):
    """Запускает cli.py в подпроцессе и стримит вывод в ноутбук."""
    cmd = [sys.executable, "cli.py", *[str(a) for a in args]]
    print(">", " ".join(cmd), flush=True)
    started = time.time()
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    for line in proc.stdout:
        if live:
            print(line.rstrip(), flush=True)
    proc.wait()
    print(f"[код возврата {proc.returncode}, {(time.time() - started) / 60:.1f} мин]")
    return proc.returncode

---
## 2. Основание гипотезы

Смотрим не на слова, а на реальные каналы. `skip_channels` — сколько каналов
энкодер отдаёт в каждый блок декодера. Ноль означает, что блок работает без
skip-connection.

In [ ]:
from experimental_tools_beliy_russak.models import describe_encoder

rows = [
    describe_encoder("tu-convnext_tiny"),
    describe_encoder("tu-convnext_small"),
    describe_encoder("tu-resnet34"),
    describe_encoder("tu-tf_efficientnetv2_s"),
]
pyramids = pd.DataFrame(rows)[
    ["encoder", "out_channels", "skip_channels", "blocks_without_skip", "encoder_m", "total_m"]
]
pyramids

Столбец `blocks_without_skip` — суть гипотезы в одном числе. Обрати внимание,
что у ConvNeXt проблема не в размере модели: у `convnext_small` она ровно та же.

---
## 3. Предпосылки

Нужны индекс датасета и фолды. Если их нет, ячейка соберёт их сама
(индексация — единственный долгий шаг, ~7 минут, делается один раз).

In [ ]:
from experimental_tools_beliy_russak.workspace import index_path, split_path

if not index_path().exists():
    run_cli(["index"])
else:
    print("индекс на месте:", index_path())

if not split_path().exists():
    run_cli(["split"])
else:
    print("фолды на месте :", split_path())

from experimental_tools_beliy_russak.splits import check_leakage, load_folds

folds = load_folds()
print("\nстрок:", len(folds), "| групп:", folds["group_id"].nunique())
print("утечка групп между фолдами:", check_leakage(folds)["leaked_groups"])
folds.groupby("fold").agg(n=("stem", "size"), негативов=("is_negative", "sum")).round(3)

---
## 4. Параметры эксперимента

Одни и те же для всех плеч. Менять их означает начать другой эксперимент —
тогда меняй и имена прогонов, иначе результаты перестанут быть сравнимыми.

In [ ]:
FOLD = 0
SEED = 42
EPOCH_SIZE = 24_000   # длина эпохи в сэмплах; данные при этом НЕ урезаются
EPOCHS = 8
VAL_FRAC = 0.5        # валидируемся на половине фолда — вдвое быстрее

RUN_A = "b1_fast"        # unet + tu-convnext_tiny
RUN_B = "b2_resnet34"    # unet + tu-resnet34
RUN_C = "b3_fpn_convnext"  # fpn  + tu-convnext_tiny  (опциональное плечо)

COMMON = [
    "-c", "baseline",
    "-s", f"train.fold={FOLD}",
    "-s", f"seed={SEED}",
    "-s", f"data.epoch_size={EPOCH_SIZE}",
    "-s", f"train.epochs={EPOCHS}",
    "-s", f"data.val_frac={VAL_FRAC}",
]

# Плечи, которые реально запускать. Обучение долгое, поэтому по умолчанию
# выключено — включай осознанно.
RUN_TRAINING = False
ARMS_TO_TRAIN = [RUN_B]   # A обычно уже обучен; C добавь, если нужен контроль

print("плечи:", RUN_A, RUN_B, RUN_C)
print("обучение включено:", RUN_TRAINING, "| запускать:", ARMS_TO_TRAIN)

---
## 5. Запуск

Каждое плечо — примерно 2.5 часа на RTX 3070. Прогоны пишутся в `runs/<имя>/`
и ничего не перетирают: при совпадении имени папка получит суффикс с меткой
времени.

Команды печатаются в любом случае — их можно скопировать и запустить в терминале,
не держа ноутбук открытым.

In [ ]:
ARM_OVERRIDES = {
    RUN_A: [],                                        # baseline как есть
    RUN_B: ["-s", "model.encoder=tu-resnet34"],
    RUN_C: ["-s", "model.arch=fpn", "-s", "model.cls_head=aux"],
}

for arm in [RUN_A, RUN_B, RUN_C]:
    args = ["train", *COMMON, "-s", f"name={arm}", *ARM_OVERRIDES[arm]]
    print(f"# {arm}\n" + " ".join([sys.executable, "cli.py", *args]) + "\n")

In [ ]:
if RUN_TRAINING:
    for arm in ARMS_TO_TRAIN:
        if (ROOT / "runs" / arm).exists():
            print(f"{arm} уже существует — пропускаю. Удали папку, чтобы переобучить.")
            continue
        run_cli(["train", *COMMON, "-s", f"name={arm}", *ARM_OVERRIDES[arm]])
else:
    print("RUN_TRAINING=False — обучение не запускается.")
    print("Поставь RUN_TRAINING=True в ячейке выше или выполни команды в терминале.")

---
## 6. Результаты

Дальше всё работает только с тем, что уже обучено. Прогоны, которых нет,
молча пропускаются.

In [ ]:
from experimental_tools_beliy_russak.analysis.compare_runs import compare_table, load_history

ARMS = [a for a in [RUN_A, RUN_B, RUN_C] if (ROOT / "runs" / a).exists()]
print("нашлось прогонов:", ARMS)
assert ARMS, "нет ни одного обученного плеча — сначала запусти обучение"

compare_table(ARMS)

Главная колонка — `AIC`. Но смотреть надо и на составляющие: выигрыш может
прийти как из `Dice_pos` (лучше сегментирует), так и из `FPR_neg` (меньше врёт
на чистых кадрах). Это разные истории, и лечатся они по-разному.

In [ ]:
from experimental_tools_beliy_russak.analysis.compare_runs import plot_history

fig = plot_history(ARMS)

---
## 7. Устойчивость к порогу

Кривая по порогу бинаризации показывает не только максимум, но и запас
прочности. Острый пик означает, что подобранный порог переобучен под этот фолд
и на тесте развалится. Пологое плато — хороший знак.

In [ ]:
from experimental_tools_beliy_russak.analysis.compare_runs import plot_threshold_curves

fig = plot_threshold_curves(ARMS, cls_threshold=0.0, min_area=0.0)

In [ ]:
# Пороги, подобранные по OOF каждого прогона (без GPU, за секунды)
from experimental_tools_beliy_russak.calibrate import calibrate_run

for arm in ARMS:
    result = calibrate_run(ROOT / "runs" / arm, top_k=3)
    print(f"\n=== {arm} ===")
    print(result["table"].to_string(index=False))

---
## 8. Качественное сравнение

Цифры говорят, что стало лучше или хуже, но не говорят почему. Сетка худших по
Dice кадров обычно отвечает за пару секунд.

In [ ]:
from IPython.display import Image, display

from experimental_tools_beliy_russak.analysis.viz import make_grid

for arm in ARMS:
    path = make_grid(ROOT / "runs" / arm, n=6, worst=True, scan=180)
    print(arm)
    display(Image(filename=str(path), width=760))

---
## 9. Выводы

Заполнить после прогона.

| Плечо | AIC | Dice_pos | FPR_neg | мин/эпоху |
|---|---|---|---|---|
| A — unet + convnext_tiny | | | | |
| B — unet + resnet34 | | | | |
| C — fpn + convnext_tiny | | | | |

**Подтвердилась ли гипотеза:**

**Откуда пришла разница (Dice или FPR):**

**Что делаем дальше:**

---

### Как продолжить

Если гипотеза подтвердилась, следующие шаги напрашиваются сами:

* энкодеры с честной картой на stride 2 — `tu-tf_efficientnetv2_s`, `tu-resnet50`;
* `unetplusplus` вместо `unet` — плотные скипы;
* больший вход (`data.size=640`) — мелкие маски перестанут теряться.

Каждый следующий эксперимент меняет **ровно один** параметр относительно
победителя и берёт те же `FOLD`, `SEED`, `EPOCH_SIZE`, `EPOCHS`. Иначе сравнивать
будет нечего.

Сводная таблица по всем прогонам вообще:

```
python cli.py board
```